# Bloc 3 — Sessió 2: Dades llestes per al model
## Correlació, codificació i escalat

A la S01 vam **explorar** les dades. Avui les deixem **preparades** perquè un model les pugui fer servir:

1. **Correlació**: quines variables es mouen juntes? Quines sobren?
2. **Codificació**: els models no llegeixen text. Com convertim les categories en números?
3. **Escalat**: per què una nota de 0 a 100 "aplasta" una variable de 0 a 1?
4. **Repte final**: deixar el dataset de la S01 (amb nuls) a punt per al Bloc 4.

Les tasques que heu de fer vosaltres estan marcades amb 📝.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("StudentsPerformance.csv")  # versió sense nuls
df.head()

---
# 1. Correlació ≠ causalitat

Totes aquestes parelles **es mouen juntes**. Però no totes per la mateixa raó.

📝 **Per parelles:** per a cada fila, decidiu si és una **causa**, si hi ha una **variable amagada** (quina?) o si és pur **atzar**.

| Parella | Causa, variable amagada o atzar? |
|---|---|
| Fumar i càncer de pulmó |  |
| Vendes de gelats i ofegaments a la platja |  |
| Nombre de bombers en un incendi i danys causats |  |
| Hores d'estudi i nota de l'examen |  |
| Número de peu i nivell de lectura (en nens) |  |
| Temperatura exterior i consum d'aire condicionat |  |
| Consum de xocolata i premis Nobel per país (r ≈ 0,79) |  |
| Pel·lícules de Nicolas Cage per any i ofegaments en piscines |  |
| `lunch = standard` i nota de mates (S01: +11 punts) |  |

> **Què és `lunch`?** El dataset és dels EUA, on el menjador escolar és **gratuït o a preu reduït** (`free/reduced`) per a les famílies amb pocs ingressos. `standard` vol dir que la família paga el preu normal. Per tant, `lunch` ens dona una pista del **nivell econòmic de la família**: no té res a veure amb el que mengen.

**Idea clau:** quan dues variables es mouen juntes, pot ser per tres motius: una causa l'altra, hi ha una **tercera variable amagada** que les mou totes dues, o és **casualitat**.

Enllaços:
- Xocolata i Nobel: l'article original es va publicar al *New England Journal of Medicine* (Messerli, 2012) — [PubMed 23050509](https://pubmed.ncbi.nlm.nih.gov/23050509/)
- Correlacions absurdes reals, com la de Nicolas Cage: [tylervigen.com/spurious-correlations](https://tylervigen.com/spurious-correlations)

# 2. Endevina la r

📝 Mira els 4 gràfics i **apunta la teva estimació** de la correlació (entre −1 i +1) **abans** d'executar la cel·la següent.

| Gràfic | La meva estimació | r real |
|---|---|---|
| A | | |
| B | | |
| C | | |
| D | | |

In [ ]:
rng = np.random.default_rng(0)
temp = rng.uniform(0, 35, 300)                                       # °C
consum = (temp - 18)**2 * 800 + 600_000 + rng.normal(0, 40_000, 300)  # forma de U (com al B01!)
x = rng.uniform(0, 10, 300)
y = -x + rng.normal(0, 3, 300)

parelles = {
    "A": (df["reading score"], df["writing score"]),
    "B": (df["math score"], df["reading score"]),
    "C": (temp, consum),
    "D": (x, y),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (nom, (a, b)) in zip(axes, parelles.items()):
    ax.scatter(a, b, s=8, alpha=0.5)
    ax.set_title(nom)
plt.show()

In [ ]:
# Revela-ho!
for nom, (a, b) in parelles.items():
    print(nom, round(np.corrcoef(a, b)[0, 1], 2))

### Datasaurus: mateixes estadístiques, gràfics diferents
13 datasets amb **la mateixa mitjana, la mateixa desviació i la mateixa correlació**... i formes totalment diferents (un és un dinosaure!).

Enllaç: [Same Stats, Different Graphs (Autodesk Research)](https://www.research.autodesk.com/publications/same-stats-different-graphs/)

La versió clàssica és el **quartet d'Anscombe** (1973). Comprova-ho tu mateix (cal connexió a internet):

In [ ]:
anscombe = sns.load_dataset("anscombe")
print(anscombe.groupby("dataset").agg(mitjana_x=("x", "mean"), mitjana_y=("y", "mean"),
                                      std_y=("y", "std")).round(2))
print(anscombe.groupby("dataset").apply(lambda d: round(d["x"].corr(d["y"]), 2)))
sns.lmplot(data=anscombe, x="x", y="y", col="dataset", col_wrap=4, ci=None, height=3)
plt.show()

> **Les estadístiques resumeixen, i resumir vol dir perdre informació. Fes sempre el gràfic.**

### Matemàtiques just in time: correlació de Pearson

$$\text{cov}(x, y) = \frac{1}{n}\sum (x_i - \bar{x})(y_i - \bar{y}) \qquad r = \frac{\text{cov}(x, y)}{\sigma_x \, \sigma_y}$$

- **+1**: pugen juntes · **−1**: una puja i l'altra baixa · **0**: cap relació **lineal**
- ⚠️ El gràfic C té r ≈ 0... i hi ha una relació claríssima! Pearson només veu **rectes**.

📝 **Deures (a mà):** hores d'estudi `x = [1, 2, 3, 4, 5]` i nota `y = [2, 4, 5, 4, 5]`. Calcula r pas a pas:
1. Les mitjanes x̄ i ȳ
2. La covariància (fes una taula amb $x_i - \bar{x}$, $y_i - \bar{y}$ i el seu producte)
3. Les desviacions $\sigma_x$ i $\sigma_y$
4. La r

**Resposta:**

# 3. Matriu de correlació

In [ ]:
# 📝 Calcula la matriu de correlació de les tres notes i dibuixa-la amb sns.heatmap
# Pista: df[cols].corr()  ·  sns.heatmap(..., annot=True, cmap="coolwarm", vmin=-1, vmax=1)
cols = ["math score", "reading score", "writing score"]


📝 **Preguntes**
- `reading` i `writing` tenen r = 0,95. Si volem predir la nota de mates, **cal mantenir les dues**?
- I `lunch`, `gender`...? Per què no surten a la matriu? (La resposta, al següent apartat.)

**Resposta:**

### Cas real: el reclutador d'Amazon
Amazon va construir una IA per filtrar currículums, entrenada amb els CV dels últims 10 anys (la majoria d'homes). La va abandonar perquè **penalitzava les dones**... sense tenir cap columna de gènere. Baixava la puntuació als CV amb la paraula *"women's"* (com *"women's chess club captain"*) i als d'universitats només femenines.

> **Esborrar una columna no esborra la informació** si altres variables hi estan correlacionades (els *proxies*). El nostre `lunch` és un proxy del nivell econòmic.

Enllaços:
- [Amazon scraps secret AI recruiting tool that showed bias against women](https://www.euronews.com/2018/10/10/amazon-scraps-secret-ai-recruiting-tool-that-showed-bias-against-women) (Reuters, 2018)
- [Fitxa del cas a l'AI Incident Database](https://incidentdatabase.ai/reports/611)

---
# 4. Codificació: els models no llegeixen text

📝 **Per parelles:** classifica les columnes.

| Columna | Nominal (sense ordre) | Ordinal (amb ordre) |
|---|---|---|
| `gender` | | |
| `race/ethnicity` | | |
| `parental level of education` | | |
| `lunch` | | |
| `test preparation course` | | |

### El parany del *label encoding*
Si convertim cada categoria en un número (A→0, B→1... E→4), el model creurà que **el grup E "val" 4 vegades el grup B**.

In [ ]:
df["race/ethnicity"].astype("category").cat.codes.value_counts().sort_index()

### Ordinal: un mapa amb l'ordre real

In [ ]:
ordre_estudis = ["some high school", "high school", "some college",
                 "associate's degree", "bachelor's degree", "master's degree"]
# 📝 Crea un diccionari {categoria: posició} i aplica'l amb .map()


### Nominal: *one-hot encoding*
Cada categoria passa a ser una columna 0/1. Amb `drop_first=True` n'eliminem una, perquè ja es dedueix de les altres.

In [ ]:
nominals = ["gender", "race/ethnicity", "lunch", "test preparation course"]
# 📝 Aplica pd.get_dummies(df, columns=nominals, drop_first=True, dtype=int)
# Quantes columnes tenies abans? I ara?


In [ ]:
# 📝 I ara SÍ: quines variables es relacionen més amb la nota de mates?
# Pista: df_cod.corr()["math score"].sort_values()


---
# 5. Escalat

### La sonda perduda a Mart
El setembre de 1999, la NASA va perdre la *Mars Climate Orbiter* quan arribava a Mart. Un equip (Lockheed Martin) donava les dades dels propulsors en **lliures-força·segon** i l'equip de navegació de la NASA esperava **newtons·segon**. Els números eren correctes; les **escales**, no.

Enllaç: [Mars Climate Orbiter (Viquipèdia)](https://ca.wikipedia.org/wiki/Mars_Climate_Orbiter)

### Qui s'assembla més a la Laia?

| Alumne | `gender_male` | `math score` |
|---|---|---|
| Laia | 0 | 70 |
| Pau | 1 | 71 |
| Marta | 0 | 80 |

📝 **A mà:** distància euclidiana $d = \sqrt{(\Delta g)^2 + (\Delta m)^2}$
1. d(Laia, Pau) = ?   d(Laia, Marta) = ?   Qui és més "semblant"?
2. Ara escala la nota amb min-max (mín. 0, màx. 100) i torna-ho a calcular.

### Matemàtiques just in time
- **Min-max:** $x' = \dfrac{x - \min}{\max - \min}$ → tot entre 0 i 1
- **Z-score (estandardització):** $z = \dfrac{x - \bar{x}}{\sigma}$ → mitjana 0 i desviació 1

**Resposta:**

In [ ]:
alumnes = pd.DataFrame({"gender_male": [0, 1, 0], "math score": [70, 71, 80]},
                       index=["Laia", "Pau", "Marta"])

def distancies(t):
    return {nom: round(float(np.linalg.norm(t.loc["Laia"] - t.loc[nom])), 2) for nom in ["Pau", "Marta"]}

print("Sense escalar:", distancies(alumnes))
# 📝 Escala "math score" amb min-max (mín. 0, màx. 100) i torna a cridar distancies()


> Sense escalar, la variable amb números més grans **decideix qui s'assembla a qui**. Els models que fan servir distàncies (**KNN, Bloc 4**) en depenen totalment.
>
> ⚠️ En un projecte real, l'escalador s'ajusta (`fit`) **només amb train** (recorda el Bloc 2).

---
# 6. Repte final: dataset llest per al model

📝 **En equips de 3:** partiu de `StudentsPerformance_with_nulls.csv` (el de la S01) i deixeu-lo **a punt per al Bloc 4**:

- [ ] Nuls tractats (decidiu l'estratègia per a cada columna)
- [ ] Ordinal codificat amb el seu ordre
- [ ] Nominals amb one-hot
- [ ] Variables redundants: decidiu-ho i justifiqueu-ho
- [ ] Notes escalades amb `MinMaxScaler`
- [ ] Desat com a `students_ready.csv`

La darrera cel·la comprova que ho heu fet bé.

In [ ]:
raw = pd.read_csv("StudentsPerformance_with_nulls.csv")
# 📝 El vostre codi aquí
ready = raw.copy()


In [ ]:
# Comprovació automàtica
assert ready.isna().sum().sum() == 0, "Encara hi ha nuls"
assert all(pd.api.types.is_numeric_dtype(t) for t in ready.dtypes), "Encara hi ha columnes de text"
assert ready.select_dtypes("number").max().max() <= 1, "Hi ha valors sense escalar"
print("Dataset llest per al model!", ready.shape)

---
# 7. Fitxa final

📝 **Respon aquí:**
1. Quina estratègia heu fet servir per a cada columna amb nuls, i per què?
2. Heu eliminat alguna variable redundant? Quina, i per què aquesta i no l'altra?
3. Quina diferència hi ha entre una variable **ordinal** i una de **nominal**? Posa'n un exemple de fora del dataset.
4. Completa: *"Abans de passar dades a un model, cal..."*

---
# Per acabar: Guess the Correlation
📝 Ara que ja en sabeu: a veure qui té millor ull! Mira el núvol de punts, estima la r i no perdis les vides.

Enllaç: [guessthecorrelation.com](https://guessthecorrelation.com/)

## Per ampliar
- Vídeo: *Pearson's Correlation, Clearly Explained* — [StatQuest a YouTube](https://www.youtube.com/results?search_query=statquest+pearson%27s+correlation+clearly+explained) (en anglès)
- Correlacions absurdes reals: [tylervigen.com/spurious-correlations](https://tylervigen.com/spurious-correlations)
- Datasaurus Dozen: [Autodesk Research](https://www.research.autodesk.com/publications/same-stats-different-graphs/)